# Module 5 — Mô hình CatBoost (Cửa hàng × Nhóm hàng × Ngày)

Dự báo `so_luong` mỗi ngày cho từng cặp cửa hàng × nhóm hàng bằng **CatBoost** (gradient boosting trên cây quyết định đối xứng, xử lý biến phân loại bằng ordered target statistics).

- **Đầu vào:** `outputs/daily_store_category.csv` (từ `module3/Module3_1_tien_xu_ly.ipynb`) + đặc trưng `tao_dac_trung` của Module 3.
- **So sánh công bằng:** cùng bảng, cùng `FEATS`, cùng mốc chia (`chia_tap`), cùng chỉ số (`danh_gia`) và baseline (`bang_baseline`) với các mô hình khác.
- **Đầu ra** `outputs/catboost/`: `cb_tuning.csv`, `cb_best_params.json`, `cb_metrics.csv`, `cb_model.cbm`, `cb_predictions.csv`, `cb_feature_importance.*`, `cb_error_by_category.csv`, `cb_error_by_store.csv`, `cb_thuc_te_vs_du_bao.png`.

**Cấu hình** (`.env` ở gốc repo, mẫu `.env.example`). **Chạy trên Google Colab:** upload `module3_preprocess.py` và `.env` cùng thư mục với notebook, rồi Runtime → Run all (ô đầu tự cài `catboost` nếu thiếu).

In [ ]:
import os, sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import ParameterSampler
try:
    from catboost import CatBoostRegressor
except ImportError:                       # Colab chưa cài sẵn catboost
    import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost"], check=True)
    from catboost import CatBoostRegressor

sys.path.append("../module3")             # module tiền xử lý dùng chung của Module 3
from module3_preprocess import (OUT_DIR as OUT, SEED, doc_bang_ngay, tao_dac_trung, chia_tap, danh_gia,
                                bang_baseline, loi_theo_nhom, nop_ket_qua, tai_ve)

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
OUT_CB = f"{OUT}/catboost"; os.makedirs(OUT_CB, exist_ok=True)
daily = doc_bang_ngay()
print(f"Bảng mô hình: {len(daily):,} dòng")

## 1. Đặc trưng và chia tập

Dùng nguyên `FEATS` của Module 3. Ba cột mã (`Mã cửa hàng_code`, `Nhóm hàng_code`, `Vùng_code`) được khai báo là **biến phân loại** của CatBoost thay vì coi là số thứ tự — đây là điểm mạnh riêng của CatBoost, không thêm thông tin mới so với các mô hình khác.

In [ ]:
d, FEATS = tao_dac_trung(daily)
CAT = ["Mã cửa hàng_code", "Nhóm hàng_code", "Vùng_code"]
fit, val, tr, te = chia_tap(d)

base_val = pd.DataFrame(bang_baseline(val)).T
rows = bang_baseline(te)
print("Baseline trên Validation:"); display(base_val)
print("Baseline trên Test:"); display(pd.DataFrame(rows).T)

## 2. Tối ưu tham số trên Validation

- **Hàm mất mát** là lựa chọn quan trọng nhất với dữ liệu thưa (≈71% ô bằng 0) và quá phân tán: thử `RMSE`, `MAE`, `Poisson`, `Tweedie` (variance_power 1,5); mỗi hàm 3 bộ `depth` / `learning_rate` / `l2_leaf_reg` ngẫu nhiên → 12 cấu hình.
- **Early stopping** theo chính hàm mất mát trên Validation (dừng sau 100 vòng không cải thiện, tối đa 2.000 vòng).
- **Chọn cấu hình theo MAE trên Validation** — cùng tiêu chí với Random Forest.

In [ ]:
LOSSES = ["RMSE", "MAE", "Poisson", "Tweedie:variance_power=1.5"]
space = {"depth": [4, 6, 8], "learning_rate": [0.03, 0.05, 0.1], "l2_leaf_reg": [1, 3, 10]}
N_MOI_LOSS, MAX_ITER, OD_WAIT = 3, 2000, 100

params = [{"loss_function": loss, **p} for loss in LOSSES
          for p in ParameterSampler(space, n_iter=N_MOI_LOSS, random_state=SEED)]
log = []
for i, p in enumerate(params):
    t = time.time()
    m = CatBoostRegressor(**p, iterations=MAX_ITER, od_type="Iter", od_wait=OD_WAIT, random_seed=SEED,
                          verbose=0, thread_count=-1, allow_writing_files=False)
    m.fit(fit[FEATS], fit["so_luong"], cat_features=CAT, eval_set=(val[FEATS], val["so_luong"]), use_best_model=True)
    s = danh_gia(val["so_luong"], np.clip(m.predict(val[FEATS]), 0, None))
    log.append({"id": i, **p, "so_vong": m.get_best_iteration() + 1, **s, "giây": round(time.time() - t, 1)})
    print(f"{i + 1:2}/{len(params)} MAE={s['MAE']:.4f} WAPE={s['WAPE']:.3f} Bias={s['Bias']:+.2f} vòng={log[-1]['so_vong']:4} {p}")
tuning = pd.DataFrame(log).sort_values("MAE").reset_index(drop=True)
tuning.to_csv(f"{OUT_CB}/cb_tuning.csv", index=False, encoding="utf-8-sig")
best = {**params[tuning.loc[0, "id"]], "iterations": int(tuning.loc[0, "so_vong"])}
with open(f"{OUT_CB}/cb_best_params.json", "w") as f:
    json.dump(best, f, indent=2)
print("Tốt nhất:", best)
# cấu hình có MAE tốt nhất của từng hàm mất mát
display(tuning.drop_duplicates("loss_function").set_index("loss_function")
        [["depth", "learning_rate", "l2_leaf_reg", "so_vong", "MAE", "RMSE", "WAPE", "Bias"]])

## 3. Train lại trên toàn bộ Train và chấm Test

Train lại cấu hình tốt nhất trên 15/01–28/02 với đúng số vòng đã chọn ở bước early stopping (không còn tập để dừng sớm), rồi chấm trên tháng 3. Dự báo âm được cắt về 0.

In [ ]:
cb = CatBoostRegressor(**best, random_seed=SEED, verbose=0, thread_count=-1, allow_writing_files=False)
cb.fit(tr[FEATS], tr["so_luong"], cat_features=CAT)
te["du_bao_cb"] = np.clip(cb.predict(te[FEATS]), 0, None)
rows["CatBoost"] = danh_gia(te["so_luong"], te["du_bao_cb"])
result = pd.DataFrame(rows).T
result["WAPE so với MA7"] = result["WAPE"] / result.loc["MA7", "WAPE"] - 1
result.to_csv(f"{OUT_CB}/cb_metrics.csv", encoding="utf-8-sig")
cb.save_model(f"{OUT_CB}/cb_model.cbm")
display(result)

## 4. Đặc trưng quan trọng và lỗi theo nhóm hàng / cửa hàng / ngày

In [ ]:
imp = pd.Series(cb.get_feature_importance(), index=FEATS).sort_values()
imp.to_csv(f"{OUT_CB}/cb_feature_importance.csv", header=["importance"], encoding="utf-8-sig")
fig, ax = plt.subplots(figsize=(7, 5)); ax.barh(imp.index, imp.values, color="#0d9488")
ax.set_title("CatBoost — mức độ quan trọng của đặc trưng"); fig.tight_layout()
fig.savefig(f"{OUT_CB}/cb_feature_importance.png", dpi=150); plt.show()

err_cat = loi_theo_nhom(te, "du_bao_cb", "Nhóm hàng")
err_store = loi_theo_nhom(te, "du_bao_cb", ["Mã cửa hàng", "Khu vực"])
err_cat.to_csv(f"{OUT_CB}/cb_error_by_category.csv", encoding="utf-8-sig")
err_store.to_csv(f"{OUT_CB}/cb_error_by_store.csv", encoding="utf-8-sig")
display(err_cat)
print(f"CatBoost có WAPE thấp hơn MA7 ở {(err_cat.WAPE < err_cat.WAPE_MA7).sum()}/{len(err_cat)} nhóm hàng "
      f"và {(err_store.WAPE < err_store.WAPE_MA7).sum()}/{len(err_store)} cửa hàng")

day = te.groupby("Ngày")[["so_luong", "du_bao_cb", "tb_7"]].sum()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(day.index, day["so_luong"], color="#1f2a2a", lw=2, label="Thực tế")
ax.plot(day.index, day["du_bao_cb"], color="#0d9488", lw=2, label="CatBoost")
ax.plot(day.index, day["tb_7"], color="#eb6834", lw=1.5, ls="--", label="MA7")
ax.set_title("Tổng số lượng tháng 3 — thực tế và dự báo"); ax.legend(); ax.grid(axis="y", alpha=.3)
fig.tight_layout(); fig.savefig(f"{OUT_CB}/cb_thuc_te_vs_du_bao.png", dpi=150); plt.show()
te[["Mã cửa hàng", "Nhóm hàng", "Khu vực", "Vùng", "Ngày", "so_luong", "du_bao_cb", "lag_1", "lag_7", "tb_7"]] \
    .to_csv(f"{OUT_CB}/cb_predictions.csv", index=False, encoding="utf-8-sig")

## 5. Kết quả và thảo luận

*Số liệu từ lần chạy ngày 09/10/2026 trên bảng mô hình chưa lọc (giữ bán buôn, online, trả lại). Chạy lại sau khi nhóm chốt bộ lọc thì cập nhật lại mục này.*

**Tối ưu tham số.** Cấu hình tốt nhất trên Validation: `loss_function="MAE"`, `depth=6`, `learning_rate=0.05`, `l2_leaf_reg=3`, 52 vòng (MAE Validation 0,745 so với MA7 0,847). Hàm mất mát quyết định kết quả nhiều hơn các tham số còn lại:

| Hàm mất mát (cấu hình có MAE tốt nhất) | MAE Val | RMSE Val | WAPE Val | Bias Val |
|---|---:|---:|---:|---:|
| MAE | **0,745** | 8,513 | **0,659** | −25,1% |
| Tweedie (1,5) | 0,853 | 8,384 | 0,754 | −11,3% |
| Poisson | 0,877 | 8,344 | 0,775 | **+5,4%** |
| RMSE | 0,984 | **8,190** | 0,870 | +5,8% |

**Kết quả trên Test (tháng 3, 67.146 ô):**

| Mô hình | MAE | RMSE | WAPE | R² | Bias |
|---|---:|---:|---:|---:|---:|
| Naive (lag 1) | 0,887 | 6,675 | 0,802 | −0,142 | −0,9% |
| Seasonal Naive (lag 7) | 0,908 | 6,728 | 0,821 | −0,160 | +1,6% |
| MA7 | 0,744 | **4,975** | 0,674 | **0,365** | **−0,2%** |
| **CatBoost** | **0,705** | 5,673 | **0,638** | 0,175 | −41,9% |

**Thảo luận.**

1. **CatBoost tốt nhất theo MAE/WAPE** — WAPE thấp hơn MA7 5,3%, thấp hơn ở 21/23 nhóm hàng và 124/130 cửa hàng. Đây là mô hình đầu tiên của nhóm vượt MA7 trên tiêu chí chọn mô hình (Random Forest: MAE 0,757, WAPE 0,686).
2. **Nhưng dự báo thiếu hệ thống 42% tổng lượng** (TB 1.392 sản phẩm/ngày toàn hệ thống so với thực tế 2.393), RMSE và R² kém MA7. Nguyên nhân: loss MAE ước lượng **trung vị**, mà 69% ô bằng 0 → mô hình dự báo gần 0 cho phần lớn ô (dự báo < 0,5 ở 95% ô bằng 0, nhưng cũng ở 41% ô có bán). Với 278 ô bán ≥ 20 sản phẩm/ngày (tổng 13.570), CatBoost chỉ dự báo 3.022, MA7 dự báo 10.904.
3. Vì vậy CatBoost **thua MA7 ở hai nhóm lớn nhất** — Điện thoại (WAPE 0,59 vs 0,56) và Phụ kiện (0,54 vs 0,51) — và thắng chủ yếu ở các nhóm bán thưa, nơi dự báo ≈ 0 đã cho sai số tuyệt đối nhỏ.
4. **Đặc trưng quan trọng:** trung bình 14 ngày (36%), trung bình 7 ngày (17%), lag 1 (16%) — mô hình chủ yếu dựa vào mức bán gần đây. `ngay_trong_thang` (12,5%) và `ngay_toi_tet` (6,6%) cao bất thường: dữ liệu train chỉ có tháng 1–2 nên hai đặc trưng này dễ học riêng giai đoạn Tết; với tháng 3 đó là ngoại suy, cần kiểm tra lại khi có thêm dữ liệu. Mã cửa hàng/vùng gần như không được dùng.
5. **Hàm ý cho nhóm:** chọn mô hình chỉ theo MAE/WAPE sẽ ưu tiên mô hình dự báo thiếu — không dùng được cho bài toán nhập/phân bổ hàng cần đúng tổng lượng. Đề xuất Module 4/5 chốt **tiêu chí chọn mô hình có xét Bias** (vd chỉ xét mô hình có |Bias| ≤ 10%, hoặc chọn theo RMSE/Poisson deviance). Theo tiêu chí đó, cấu hình Tweedie/Poisson của CatBoost (Bias Validation −11% / +5%; MAE Validation 0,853 / 0,877, xấp xỉ MA7 0,847) là ứng viên phù hợp hơn cấu hình MAE.

**Giới hạn:** bảng mô hình chưa lọc bán buôn/online (ô lớn nhất 1.274 sản phẩm), chỉ 12 cấu hình được thử, một mốc chia duy nhất (chưa backtest nhiều tháng).

## Nộp kết quả vào báo cáo và tải về (Colab)

`nop_ket_qua` lọc các kết quả đưa vào báo cáo sang `bao_cao/du_lieu/catboost/`. Sau đó chạy `python module5/tao_bao_cao.py` để cập nhật `bao_cao/BAO_CAO.md` rồi commit `bao_cao/` (trên Colab: giải nén `bao_cao_catboost.zip` vào `bao_cao/du_lieu/catboost/` trong repo trước).

In [ ]:
nop_ket_qua("catboost", val)
tai_ve(OUT_CB, "module5_catboost_outputs")